In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from sklearn.naive_bayes import MultinomialNB,BernoulliNB
from sklearn.metrics import f1_score,accuracy_score,precision_score,recall_score
import pickle

In [17]:
df = pd.read_csv("/workspaces/Elreno23-machine-learning-python-template/data/raw/07-playstore_reviews.csv")
df
df["review"][0]

" privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only. if only i wanna do on facebook is to read on my newsfeed and just wanna response on message i want to. pls reconsidered my review. i tried to turn off chat but still can see me as online."

In [18]:
df = df.drop(columns="package_name")
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


## Procesamiento de texto

### Eliminar espacios y convertir a minusculas el texto


In [19]:
df["review"] = df["review"].str.strip().str.lower()

In [20]:
X = df.drop(columns="polarity")
y = df["polarity"]

In [21]:
X_train, X_test, y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#stratify: Asegura que el split(train y test) tengan la misma distribucion de clases que el df original


In [22]:
vec_model = CountVectorizer(stop_words="english") #Contamos cuantas veces aparece cada palabra
X_train = vec_model.fit_transform(X_train["review"]).toarray()
X_test = vec_model.transform(X_test["review"]).toarray()
X_train           

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3272))

### Conclusiones

Al pasar el df(df con una sola variable) completo al fit y transform de esta manera (X_train) asume que es una lista con un objeto que contiene las 712 reseñas, es decir, un solo documento con todas las reseñas.

al pasar el df(df["review"]) le pasamos una serie con 712 strings, cada reseña es un documento independiente, lo cual es correcto.

In [23]:
multi = MultinomialNB()
multi.fit(X_train, y_train)

bernou = BernoulliNB()
bernou.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [24]:
#Rejilla de parametros
params_multi = {'alpha': np.linspace(0.01, 2.0, 50),
          'fit_prior': [True, False]}
        
random_search_multi = RandomizedSearchCV(multi,params_multi, n_iter=20, cv=5, scoring="accuracy", random_state=42)
random_search_multi.fit(X_train, y_train)

random_search_multi.best_estimator_,random_search_multi.best_score_


(MultinomialNB(alpha=np.float64(1.6751020408163266), fit_prior=False),
 np.float64(0.8019797104304146))

In [25]:
params_bernou = {"alpha": np.linspace(0.01, 2.0, 50),
                 "fit_prior": [True, False],
                 "binarize": np.linspace(0.0, 1.0, 10)}  # umbrales entre 0 y 1
        
random_search_bernoulli = RandomizedSearchCV(bernou, params_bernou, n_iter=20, cv=5, scoring="accuracy", random_state=42)
random_search_bernoulli.fit(X_train, y_train)
random_search_bernoulli.best_params_,random_search_bernoulli.best_score_

({'fit_prior': True,
  'binarize': np.float64(0.8888888888888888),
  'alpha': np.float64(0.13183673469387755)},
 np.float64(0.789382448537378))

In [26]:

y_predict_test_multi = random_search_multi.predict(X_test)
y_predict_train_multi = random_search_multi.predict(X_train) #Comprobamos overfitting
y_predict_test_multi, y_predict_train_multi


(array([0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1,
        1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1,
        1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0]),
 array([1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1,
        1, 1, 0, 1,

In [27]:
y_predict_test_bernou = random_search_bernoulli.predict(X_test)
y_predict_train_bernou = random_search_bernoulli.predict(X_train) #Comprobamos overfitting
y_predict_test_bernou, y_predict_train_bernou


(array([0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
        0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1,
        1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
        1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
        1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0]),
 array([1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1,
        1, 1, 0, 1,

In [28]:
def get_classifier_metrics(y_predict_test, y_test, y_predict_train, y_train, average='micro'):
    metrics_train = (accuracy_score(y_train, y_predict_train),
                     f1_score(y_train, y_predict_train, average=average),
                     precision_score(y_train, y_predict_train, average=average),
                     recall_score(y_train, y_predict_train, average=average))
    metrics_test = (accuracy_score(y_test, y_predict_test),
                    f1_score(y_test, y_predict_test, average=average),
                    precision_score(y_test, y_predict_test, average=average),
                    recall_score(y_test, y_predict_test, average=average))
    return pd.DataFrame(data=[metrics_train, metrics_test],
                        columns=['Accuracy', 'F1 Score', 'Precision', 'Recall'],
                        index=['Train set', 'Test set'])

result_multi = get_classifier_metrics(y_predict_test_multi, y_test, y_predict_train_multi, y_train)
result_multi

,Accuracy,F1 Score,Precision,Recall
Train set,0.942416,0.942416,0.942416,0.942416
Test set,0.871508,0.871508,0.871508,0.871508


In [29]:
result_bernou = get_classifier_metrics(y_predict_test_bernou, y_test, y_predict_train_bernou, y_train)
result_bernou

,Accuracy,F1 Score,Precision,Recall
Train set,0.980337,0.980337,0.980337,0.980337
Test set,0.854749,0.854749,0.854749,0.854749


In [30]:
with open('../models/multinomial-naive-bayes-playstore-reviews.pkl', 'wb') as f:
    pickle.dump(multi, f)
with open('../models/countvectorizer-multinomial-naive-bayes-playstore-reviews.pkl', 'wb') as f:
    pickle.dump(vec_model, f)

### Conclusiones

-El modelo MultinomialNB logra mayor precisión que BernoulliNB, y esto se explica por cómo funcionan internamente. Ambos modelos convierten todas las palabras de las reseñas en “columnas” dentro de una matriz.

-MultinomialNB cuenta la frecuencia de cada palabra en la reseña: si aparece “excelente” dos veces, en la columna correspondiente se registra un 2. De esta forma, aprovecha la cantidad de veces que se repite una palabra para decidir si la reseña es positiva o negativa.

-BernoulliNB, en cambio, solo registra la presencia o ausencia de cada palabra: si “excelente” aparece una o varias veces, simplemente coloca un 1 en la columna.

-El problema surge cuando una reseña contiene palabras contradictorias, como “excelente” y “malo”. BernoulliNB marcará un 1 en ambas columnas y dependerá del entrenamiento decidir la clase. Si el modelo aprendió que “excelente” aparece más en reseñas positivas, puede inclinarse hacia positivo; si “malo” pesa más en negativas, puede inclinarse hacia negativo.

-Por eso en estos datasets de reseñas, MultinomialNB suele predecir con mayor precisión(como observamos en las metricas), porque la frecuencia de las palabras aporta información que BernoulliNB no toma en cuenta.